# Season Holdout — Gridsearch + Backtest, with vs. without Google Trends (2025/26, W40–W20)

Runs the full pipeline — gridsearch on pre-season data, then a vintage-aware
backtest over the held-out 2025/26 season (ISO W40–W20), scored against known
truth — **twice**: once with Google Trends features, once without. Same
locations, same season, same everything else, so the only thing that differs
between the two runs is the GT features — making the WIS/AE comparison a clean
answer to "does adding Google Trends help."

- **Test set**: 2025/26 season, ISO week 40 (2025-10-05) through ISO week 20
  (2026-05-17) — completely untouched by the gridsearch, for both variants.
- **Gridsearch period**: everything before the test season; `CUTOFF_Q` (75%
  train / 25% validation) splits *within* that pre-season data, same as before.
- **Data handling**: gridsearch uses full/current data (no revision-bias
  concern for a fully-settled pre-season period); the season evaluation is
  vintage-aware (`resolve_long_timeseries_asof`, same as
  `forecast_backtest.py --snapshots-only`) so the WIS/AE reflects genuine live
  performance, not numbers inflated by later revisions.

## Setup

In [1]:
# Optional: install dependencies (uncomment if running in a fresh env / Colab)
# %pip install "lightgbm>=4.3" "lightgbmlss>=0.2.5" "optuna>=3.6" "matplotlib>=3.8"


In [2]:
from pathlib import Path
import sys
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "src" / "gridsearch_lgbm.py").exists():
            return p
    raise FileNotFoundError(
        "Could not locate repo root (src/gridsearch_lgbm.py not found above "
        f"{start}). Run this notebook from inside the MIGHTE-respicast-jointGBM checkout."
    )

ROOT_DIR = find_repo_root(Path.cwd())
print("Repo root:", ROOT_DIR)


Repo root: /home/nadillia/Documents/MIGHTE-respicast-jointGBM


In [3]:
sys.path.insert(0, str(ROOT_DIR / "src"))

%load_ext autoreload
%autoreload 2

import gridsearch_lgbm as gs
from model_joint_twostage_eu import RuntimeConfig, run_prospective
from forecast_backtest import choose_backtest_origins, target_slug
from build_long_timeseries import resolve_long_timeseries, resolve_long_timeseries_asof

print("import OK")


/home/nadillia/Documents/MIGHTE-respicast-jointGBM/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


import OK


In [4]:
# Hub repo checkout (target-data + locations/forecasting-weeks files). Pulls if already cloned.
HUB_DIR = ROOT_DIR / "RespiCast-SyndromicIndicators"
LOCATIONS_FILE = HUB_DIR / "supporting-files" / "locations_iso2_codes.csv"
FORECASTING_WEEKS_FILE = HUB_DIR / "supporting-files" / "forecasting_weeks.csv"

if not LOCATIONS_FILE.exists():
    !git clone --depth 1 https://github.com/european-modelling-hubs/RespiCast-SyndromicIndicators.git "{HUB_DIR}"
else:
    !git -C "{HUB_DIR}" pull --ff-only


remote: Enumerating objects: 87, done.
remote: Counting objects: 100% (87/87), done.
remote: Compressing objects: 100% (48/48), done.
remote: Total 70 (delta 52), reused 37 (delta 22), pack-reused 0 (from 0)
Unpacking objects: 100% (70/70), 2.82 MiB | 485.00 KiB/s, done.
From https://github.com/european-modelling-hubs/RespiCast-SyndromicIndicators
   5963f36..b149fa1  main       -> origin/main
Updating 5963f36..b149fa1
Fast-forward
 .github/data-storage/changes_db.json               |     75 +-
 .github/data-storage/ensemble_db.json              |     11 +-
 .github/data-storage/target_db.json                |     15 +-
 ...8-19-respicast-hubEnsemble-ensemble_models.json |      2 +-
 model-evaluation/latest-forecast_scores.csv        |   2433 +-
 .../snapshots/2026-08-12-forecast_scores.csv       | 258628 ++++++++++++++++++
 .../ISI-LightGBM/2026-08-19-ISI-LightGBM.csv       |   3497 +
 .../2026-08-19-respicast-hubEnsemble.csv           |   6637 +-
 .../2026-08-19-respicast-quantileBas

In [5]:
# Full current canonical data (not vintage-limited) -- fine here, the whole test
# season is already in the past, so there's nothing left to "revise" by watching it.
DATA_FILE = ROOT_DIR / "data" / "processed" / "respicast_long_latest.csv"

canonical = resolve_long_timeseries(HUB_DIR)
DATA_FILE.parent.mkdir(parents=True, exist_ok=True)
canonical.to_csv(DATA_FILE, index=False)
print(f"Refreshed {DATA_FILE}: {len(canonical)} rows, "
      f"max truth_date={pd.to_datetime(canonical['truth_date']).max().date()}")


Refreshed /home/nadillia/Documents/MIGHTE-respicast-jointGBM/data/processed/respicast_long_latest.csv: 22357 rows, max truth_date=2026-08-09


## Config

In [6]:
TARGET = "ILI"  # or "ARI"

# Season-holdout-safe GT file (denoise/detrend fit only on pre-season data, replayed
# onto the test season without re-fitting -- see google_preprocessing/notebooks/
# main_season_holdout.ipynb). NOT the operational google_trends_preprocessed.csv,
# which fits on the whole history and would leak 2025/26-and-beyond information
# into every pre-season feature value.
GT_FILE = ROOT_DIR / "google_preprocessing" / "data" / "processed" / "google_trends_preprocessed_season_holdout.csv"

# Same pre-season-safe keyword/cluster selection as GT_FILE, but skips the denoise/detrend
# fit entirely (no fitting step at all, so no train/test split needed for this one) -- just
# the raw search-volume values. Used only by the gridsearch-only 3-way comparison further down.
RAW_GT_FILE = ROOT_DIR / "google_preprocessing" / "data" / "processed" / "google_trends_raw_season_holdout.csv"

EXCLUDE_COVID = True
INCLUDE_GT_LEAD = True   # only applies to the gt_proc/gt_raw variants
GT_MIN_CORR = 0.3        # only applies to the gt_proc/gt_raw variants

SEED = 4321
CUTOFF_Q = 0.75  # train/validation split *within* the pre-season gridsearch period

# Moderate budget, not the full 70/60 from Gridsearch_LightGBM.ipynb -- this runs the
# search twice (once per GT variant), so full-strength trial counts would mean hours
# per variant. TPE gets most of its improvement in the first ~20-30 trials on a search
# space this size (3-4 dims); past that it's mostly polishing. Bump back up for a final,
# for-the-paper run once you're happy with everything else.
N_TRIALS_STAGE1 = 30
N_TRIALS_STAGE2 = 25

OWN_LAGS = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 26, 52]
DONOR_LAGS = [1, 2, 3, 4, 8, 12]
DONOR_TOP_K = 4
OTHER_TOP_K = 2

# -- Test season: 2025/26, ISO week 40 through ISO week 20 --
TEST_SEASON_START = pd.Timestamp.fromisocalendar(2025, 40, 7)  # Sunday of ISO week 40, 2025
TEST_SEASON_END   = pd.Timestamp.fromisocalendar(2026, 20, 7)  # Sunday of ISO week 20, 2026

# Gridsearch anchor: the last week strictly BEFORE the test season. build_matrix's
# cutoff_date truncation means nothing from TEST_SEASON_START onward ever enters
# the gridsearch (own_lags/build_pooled_examples can't leak past this either --
# there's simply no row past this date for a horizon shift to reach into).
GRIDSEARCH_ANCHOR = TEST_SEASON_START - pd.Timedelta(weeks=1)

# Vintage-aware test evaluation: each test-season origin only sees dated snapshot
# data that genuinely existed by that origin (same mechanism as
# forecast_backtest.py --snapshots-only). Set False to use full/current data at
# every origin instead (faster, but the resulting WIS/AE would be optimistically
# biased by later revisions to early-season truth).
SNAPSHOTS_ONLY_FOR_TEST = True

# Keep this small (e.g. 3-5) while debugging the loop -- bump up for the real run.
NUM_BAGS = 25

OTHER = "ARI" if TARGET == "ILI" else "ILI"
TARGET_COL, OTHER_COL = gs.TARGET_COLUMN[TARGET], gs.TARGET_COLUMN[OTHER]

# -- GT variants for the gridsearch + season-backtest comparison above: name -> gt_file
# (None = no Google Trends). gt_raw is intentionally NOT in here -- it only gets the
# gridsearch-only comparison further down, not the full season backtest. --
GT_VARIANTS = {
    "no_gt": None,
    "gt_proc": GT_FILE,
}

# Same location set for every variant, so the comparison isn't confounded by
# different countries being included/excluded between runs.
LOCATIONS = sorted(pd.read_csv(GT_FILE, usecols=["location"])["location"].astype(str).unique().tolist())
if TARGET == "ILI":
    LOCATIONS = [loc for loc in LOCATIONS if loc not in {"CY", "IT"}]  # currently no ILI data for these countries

SUBMISSION_DIR = ROOT_DIR / "forecasts" / "retrospective" / "submission" / "season_2025_26_holdout"
MODEL_TAG = "SeasonHoldout"

print(f"target={TARGET_COL!r}  secondary={OTHER_COL!r}  locations={len(LOCATIONS)}")
print(f"Gridsearch uses data up to  : {GRIDSEARCH_ANCHOR.date()}  (train {1-CUTOFF_Q:.0%}/valid split within this)")
print(f"Test season (held out)      : {TEST_SEASON_START.date()} to {TEST_SEASON_END.date()}")
print(f"Test evaluation vintage-aware: {SNAPSHOTS_ONLY_FOR_TEST}")
print(f"Variants to run (gridsearch + season backtest): {list(GT_VARIANTS)}")


target='ILI incidence'  secondary='ARI incidence'  locations=30
Gridsearch uses data up to  : 2025-09-28  (train 25%/valid split within this)
Test season (held out)      : 2025-10-05 to 2026-05-17
Test evaluation vintage-aware: True
Variants to run (gridsearch + season backtest): ['no_gt', 'gt_proc']


### Parallelism settings (gridsearch only — nothing else in the repo is affected)

Optuna can run multiple gridsearch trials at once (`N_JOBS_TRIALS`), and each trial's
LightGBM fit can be capped to fewer threads (`LGB_NUM_THREADS`) so the concurrent
trials don't fight each other for cores. Total concurrent CPU usage is roughly
`N_JOBS_TRIALS * LGB_NUM_THREADS`, kept under `CPU_BUDGET_FRACTION` of the machine.

- **`os.cpu_count()` cannot be trusted on a shared/managed server.** It reports the
  *host's* total core count, not your actual quota. A Kubernetes/JupyterHub-style
  allocation of e.g. `0.1 CPU` can still show `os.cpu_count()` in the dozens — asking
  for even a handful of threads on that quota gets the OS's thread-creation syscall
  refused outright: `libgomp: Thread creation failed: Resource temporarily
  unavailable`, which kills the kernel with no Python traceback (looks exactly like
  the crash screenshot from earlier). The cell below tries to read the real cgroup
  quota first and falls back to `os.cpu_count()` only if that's unavailable; if it
  still gets it wrong on your setup, set `CPU_BUDGET_OVERRIDE` to your actual quota
  directly instead of trusting auto-detection.
- **If your quota works out under 1 full core** (e.g. that `0.1 CPU` case), there is no
  real parallel headroom to exploit at all — the cell forces `N_JOBS_TRIALS = 1` and
  `LGB_NUM_THREADS = 1` automatically in that case (fully sequential, single-threaded;
  note `1`, not `0` — `0` means "let LightGBM auto-detect and use all cores," which is
  exactly what fails here).
- **Your own laptop**: lower `CPU_BUDGET_FRACTION` (e.g. `0.3`–`0.4`), or just set
  `N_JOBS_TRIALS = 1` to force sequential trials — the notebook's original behavior. A
  laptop's cooling is much weaker than a server's: pinning most cores for a
  multi-hour gridsearch risks sustained thermal throttling (the CPU clocks itself
  down once it overheats, which can make the run *slower*, not faster), plus it runs
  hot and loud the whole time.
- This section's settings only reach the two gridsearch loops in *this* notebook
  (`run_variant` above and `run_gridsearch_only` below) — `Gridsearch_LightGBM.ipynb`
  and the forecast/backtest bagging loop are untouched, still `n_jobs=1`,
  `num_threads=0` ("all cores" for whichever single fit is running at a time), exactly
  as before.

In [5]:
import os


def detect_cpu_quota() -> float:
    """Best-effort real CPU allocation, in fractional cores -- checks the cgroup
    quota (v2, then v1) before falling back to os.cpu_count(), since the latter
    reports the host's total cores, not a container/shared-server allocation."""
    try:  # cgroup v2
        with open("/sys/fs/cgroup/cpu.max") as f:
            quota_str, period_str = f.read().split()
        if quota_str != "max":
            return int(quota_str) / int(period_str)
    except (FileNotFoundError, ValueError):
        pass
    try:  # cgroup v1
        with open("/sys/fs/cgroup/cpu/cpu.cfs_quota_us") as f:
            quota = int(f.read())
        with open("/sys/fs/cgroup/cpu/cpu.cfs_period_us") as f:
            period = int(f.read())
        if quota > 0:
            return quota / period
    except (FileNotFoundError, ValueError):
        pass
    return float(os.cpu_count() or 1)


# If detect_cpu_quota() still doesn't match what your server/provider actually
# allots you, skip auto-detection and hardcode it here instead, e.g. 0.1.
CPU_BUDGET_OVERRIDE = None

AVAILABLE_CPUS = CPU_BUDGET_OVERRIDE if CPU_BUDGET_OVERRIDE is not None else detect_cpu_quota()

# Fraction of that budget you're willing to use -- see the note above for
# shared-server vs. laptop guidance.
CPU_BUDGET_FRACTION = 0.5

TOTAL_THREAD_BUDGET = AVAILABLE_CPUS * CPU_BUDGET_FRACTION

if TOTAL_THREAD_BUDGET < 1:
    # Under one full core available: no real parallel headroom. Force fully
    # sequential, single-threaded -- this is what avoids "libgomp: Thread
    # creation failed: Resource temporarily unavailable".
    N_JOBS_TRIALS = 1
    LGB_NUM_THREADS = 1
else:
    N_JOBS_TRIALS = min(4, int(TOTAL_THREAD_BUDGET))                     # concurrent Optuna trials
    LGB_NUM_THREADS = max(1, int(TOTAL_THREAD_BUDGET) // N_JOBS_TRIALS)  # threads per trial's LightGBM fit

print(f"{AVAILABLE_CPUS:.2f} CPUs detected -> budget {TOTAL_THREAD_BUDGET:.2f} threads "
      f"({CPU_BUDGET_FRACTION:.0%} of the machine) "
      f"-> N_JOBS_TRIALS={N_JOBS_TRIALS} x LGB_NUM_THREADS={LGB_NUM_THREADS}")


8.00 CPUs detected -> budget 4.00 threads (50% of the machine) -> N_JOBS_TRIALS=4 x LGB_NUM_THREADS=1


## Test-season origins + truth (shared across variants)

Computed once — identical for both GT variants.

In [7]:
max_truth_date = pd.to_datetime(canonical["truth_date"], errors="coerce").max()

test_origins = choose_backtest_origins(
    forecasting_weeks_file=FORECASTING_WEEKS_FILE,
    max_truth_date=max_truth_date,
    start_origin_date=TEST_SEASON_START,
    end_origin_date=TEST_SEASON_END,
    max_origins=None,
)
print(f"{len(test_origins)} test-season origins, "
      f"{test_origins['origin_date'].min().date()} to {test_origins['origin_date'].max().date()}")

truth = pd.read_csv(DATA_FILE)
truth = truth[truth["target"] == TARGET_COL][["location", "truth_date", "value"]].copy()
truth["truth_date"] = pd.to_datetime(truth["truth_date"])
truth = truth.rename(columns={"truth_date": "target_end_date", "value": "actual"})

if SNAPSHOTS_ONLY_FOR_TEST:
    vintage_dir = DATA_FILE.parent / "vintage"
    vintage_dir.mkdir(parents=True, exist_ok=True)

test_origins.head()


32 test-season origins, 2025-10-08 to 2026-05-13


,origin_date,target_end_date
0,2025-10-08,2025-09-28
1,2025-10-15,2025-10-05
2,2025-10-22,2025-10-12
3,2025-10-29,2025-10-19
4,2025-11-05,2025-10-26


## Pipeline function

One function does gridsearch (pre-season) → season backtest (vintage-aware) →
scoring, for one GT variant. Called once per entry in `GT_VARIANTS` below —
everything it returns (studies, feature matrix, predictions, scored table)
stays inspectable afterward via the `results` dict, so this is still a
debuggable step-by-step run, just parameterized by variant instead of
copy-pasted per variant.

In [ ]:
def run_variant(variant_name: str, gt_file):
    use_gt = gt_file is not None
    print(f"\n{'='*72}\nVARIANT: {variant_name}  (google_trends={'yes' if use_gt else 'no'})\n{'='*72}")

    # -- Gridsearch feature matrix (pre-season only) --
    df, y, feat_cols = gs.build_matrix(
        DATA_FILE, TARGET_COL, OTHER_COL, LOCATIONS, GRIDSEARCH_ANCHOR,
        EXCLUDE_COVID, OWN_LAGS, DONOR_LAGS, DONOR_TOP_K, OTHER_TOP_K,
        gt_file=str(gt_file) if use_gt else None,
        include_gt_lead=(INCLUDE_GT_LEAD if use_gt else False),
        gt_min_corr=GT_MIN_CORR,
    )
    print(f"[{variant_name}] matrix: {df.shape[0]} rows, {len(feat_cols)} features")
    assert df["date"].max() < TEST_SEASON_START, "Gridsearch data reaches into the test season!"

    # -- Stage 1 (RMSE) + Stage 2 (WIS) --
    study_s1 = gs.run_stage1_study(variant_name, df, y, feat_cols, SEED, N_TRIALS_STAGE1, CUTOFF_Q, verbose=True,
                                    num_threads=LGB_NUM_THREADS, n_jobs=N_JOBS_TRIALS)
    stage1_best = {**study_s1.best_params, "rounds": study_s1.best_trial.user_attrs["best_round"]}
    study_s2 = gs.run_stage2_study(variant_name, df, y, feat_cols, stage1_best, SEED, N_TRIALS_STAGE2, CUTOFF_Q, verbose=True,
                                    num_threads=LGB_NUM_THREADS, n_jobs=N_JOBS_TRIALS)
    tuned_params = gs.pack(study_s1, study_s2)

    params_path = ROOT_DIR / f"best_params_season2025_26_holdout_{TARGET.lower()}_{variant_name}.json"
    with open(params_path, "w") as f:
        json.dump({variant_name: tuned_params}, f, indent=2)
    print(f"[{variant_name}] saved tuned params to {params_path}")

    # -- Season backtest --
    sub_dir = SUBMISSION_DIR / variant_name
    sub_dir.mkdir(parents=True, exist_ok=True)

    all_preds = []
    for i, row in enumerate(test_origins.itertuples(index=False), start=1):
        origin_date = pd.to_datetime(row.origin_date)
        anchor_date = pd.to_datetime(row.target_end_date)

        if SNAPSHOTS_ONLY_FOR_TEST:
            origin_data_file = vintage_dir / f"{origin_date.date()}-respicast_long.csv"
            if not origin_data_file.exists():
                resolve_long_timeseries_asof(HUB_DIR, origin_date).to_csv(origin_data_file, index=False)
        else:
            origin_data_file = DATA_FILE

        cfg = RuntimeConfig(
            data_file=origin_data_file,
            target=TARGET_COL,
            output=sub_dir / f"tmp_{target_slug(TARGET_COL)}.csv",
            locations_file=LOCATIONS_FILE,
            forecasting_weeks_file=FORECASTING_WEEKS_FILE,
            max_horizons=4,
            num_bags=NUM_BAGS,
            bag_frac=0.7,
            location_bag_frac=1.0,
            location_bag_min=1,
            seed=SEED,
            stage1_rounds=tuned_params["rounds"],
            stage2_rounds=tuned_params["stage2_rounds"],
            own_lags=OWN_LAGS,
            donor_lags=DONOR_LAGS,
            donor_top_k=DONOR_TOP_K,
            other_top_k=OTHER_TOP_K,
            min_overlap=30,
            min_train_rows=800,
            target_mode="delta_log",
            sigma_mode="bounded",
            recent_weeks_required=4,
            anchor_date=anchor_date,
            origin_date=origin_date,
            google_trends_file=Path(gt_file) if use_gt else None,
            num_leaves=tuned_params["num_leaves"],
            learning_rate=tuned_params["learning_rate"],
            min_child_samples=tuned_params["min_child_samples"],
            feature_fraction=tuned_params["feature_fraction"],
            lambda_l2=tuned_params["lambda_l2"],
            s2_min_child_samples=tuned_params["s2_min_child_samples"],
            exclude_covid=EXCLUDE_COVID,
            include_gt_lead=(INCLUDE_GT_LEAD if use_gt else False),
            gt_min_corr=GT_MIN_CORR,
        )

        pred = run_prospective(cfg)
        if pred.empty:
            raise RuntimeError(f"[{variant_name}] no forecasts generated for origin={origin_date.date()}")
        pred.to_csv(sub_dir / f"{origin_date.date()}-{MODEL_TAG}-{variant_name}.csv", index=False)
        all_preds.append(pred)
        print(f"[{variant_name}] [{i}/{len(test_origins)}] origin={origin_date.date()} rows={len(pred)}")

    all_preds = pd.concat(all_preds, ignore_index=True)
    all_preds["target_end_date"] = pd.to_datetime(all_preds["target_end_date"])

    # -- Score against known truth --
    scored = all_preds.merge(truth, on=["location", "target_end_date"], how="inner")
    piv = scored.pivot_table(
        index=["origin_date", "location", "horizon", "target_end_date", "actual"],
        columns="output_type_id", values="value",
    ).reset_index()
    quantile_cols = sorted(c for c in piv.columns if isinstance(c, float))
    piv["wis"] = gs.wis_vectorized(piv[quantile_cols].to_numpy(), piv["actual"].to_numpy(), np.array(quantile_cols))
    piv["ae"] = (piv[0.5] - piv["actual"]).abs()

    print(f"[{variant_name}] overall mean WIS = {piv['wis'].mean():.3f}   mean AE = {piv['ae'].mean():.3f}")

    return {
        "df": df, "feat_cols": feat_cols,
        "study_s1": study_s1, "study_s2": study_s2, "tuned_params": tuned_params,
        "all_preds": all_preds, "piv": piv,
    }


## Run both variants

In [9]:
results = {}
for variant_name, gt_file in GT_VARIANTS.items():
    results[variant_name] = run_variant(variant_name, gt_file)



VARIANT: no_gt  (google_trends=no)
[no_gt] matrix: 33869 rows, 100 features
=== Stage 1 (RMSE) optimizing: no_gt ===


Best trial: 22. Best value: 636.441: 100%|██████████| 30/30 [07:43<00:00, 15.44s/it]


=== Stage 2 (WIS) optimizing: no_gt  (stage1 rounds frozen at 1000) ===


  0%|          | 0/25 [00:00<?, ?it/s]/home/nadillia/Documents/MIGHTE-respicast-jointGBM/.venv/lib/python3.12/site-packages/lightgbmlss/utils.py:19: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  nan=float(torch.nanmean(predt)),


*** Using GaussianFrozenLocBounded (frozen mu, sigma in [0.1, 0.6]) ***


Best trial: 23. Best value: 105.821: 100%|██████████| 25/25 [08:49<00:00, 21.18s/it]


[no_gt] saved tuned params to /home/nadillia/Documents/MIGHTE-respicast-jointGBM/best_params_season2025_26_holdout_ili_no_gt.json
[ILI incidence] recent-truth filter: require >= 2025-08-31 (4 weeks), eligible=18, excluded=17
[ILI incidence] excluded_locations=['AT', 'BG', 'CH', 'CY', 'DE', 'FR', 'GB-ENG', 'GB-NIR', 'GB-SCT', 'GB-WLS', 'HU', 'IT', 'LI', 'LV', 'PT', 'SE', 'SK']
[ILI incidence] anchor=2025-09-28 train_rows=26402 test_rows=72
[ILI incidence] season_bag_size=6/9 location_bag_size=18/18
[ILI incidence] successful_bags=25
[ILI incidence] per-location bag coverage: min=25 median=25.0 max=25
[no_gt] [1/32] origin=2025-10-08 rows=1656
[ILI incidence] recent-truth filter: require >= 2025-09-07 (4 weeks), eligible=19, excluded=16
[ILI incidence] excluded_locations=['AT', 'BG', 'CH', 'CY', 'DE', 'GB-ENG', 'GB-NIR', 'GB-SCT', 'GB-WLS', 'HU', 'IT', 'LI', 'LV', 'PT', 'SE', 'SK']
[ILI incidence] anchor=2025-10-05 train_rows=27552 test_rows=76
[ILI incidence] season_bag_size=6/9 locatio

KeyboardInterrupt: 

## Compare — with vs. without Google Trends

In [ ]:
comparison = pd.DataFrame({
    name: {
        "mean_wis": r["piv"]["wis"].mean(),
        "mean_ae": r["piv"]["ae"].mean(),
        "n": len(r["piv"]),
    }
    for name, r in results.items()
}).T
if "no_gt" in comparison.index and "gt_proc" in comparison.index:
    comparison["wis_pct_change_vs_no_gt"] = 100 * (comparison["mean_wis"] - comparison.loc["no_gt", "mean_wis"]) / comparison.loc["no_gt", "mean_wis"]
comparison


In [ ]:
per_horizon = pd.concat(
    {name: r["piv"].groupby("horizon")[["wis", "ae"]].mean() for name, r in results.items()},
    axis=1,
)
per_horizon


In [ ]:
plt.figure(figsize=(7, 4))
for name, r in results.items():
    h = r["piv"].groupby("horizon")["wis"].mean()
    plt.plot(h.index, h.values, marker="o", label=name)
plt.xlabel("Horizon (weeks ahead)")
plt.ylabel("Mean WIS (lower = better)")
plt.title(f"{TARGET_COL} — 2025/26 season holdout: with vs. without Google Trends")
plt.xticks(per_horizon.index)
plt.legend()
plt.tight_layout()
plt.show()


## 3-way gridsearch-only comparison — no GT / GT processed / GT raw

Everything below only runs the gridsearch (pre-season, same as above) for a third
variant, `gt_raw` — the same pre-season-safe keyword/cluster selection as `gt_proc`,
but skipping the denoise/detrend fit entirely (raw search-volume values). **No season
backtest for any of the three variants here** — this section answers "which config
tunes best on the pre-season data," not "which forecasts the held-out season best"
(that's what the with/without comparison above already covers for `no_gt`/`gt_proc`).

Reruns the gridsearch for `no_gt`/`gt_proc` too (not reusing `results` from above) so
all three summary numbers/plots come from the same code path and are directly
comparable.

In [ ]:
import lightgbm as lgb

GT_VARIANTS_3WAY = {"no_gt": None, "gt_proc": GT_FILE, "gt_raw": RAW_GT_FILE}
COLORS_3WAY = {"no_gt": "#377EB8", "gt_proc": "#4DAF4A", "gt_raw": "#E41A1C"}


def run_gridsearch_only(variant_name: str, gt_file):
    use_gt = gt_file is not None
    print(f"\n{'='*72}\nGRIDSEARCH ONLY: {variant_name}  (google_trends={'yes' if use_gt else 'no'})\n{'='*72}")

    df, y, feat_cols = gs.build_matrix(
        DATA_FILE, TARGET_COL, OTHER_COL, LOCATIONS, GRIDSEARCH_ANCHOR,
        EXCLUDE_COVID, OWN_LAGS, DONOR_LAGS, DONOR_TOP_K, OTHER_TOP_K,
        gt_file=str(gt_file) if use_gt else None,
        include_gt_lead=(INCLUDE_GT_LEAD if use_gt else False),
        gt_min_corr=GT_MIN_CORR,
    )
    print(f"[{variant_name}] matrix: {df.shape[0]} rows, {len(feat_cols)} features")
    assert df["date"].max() < TEST_SEASON_START, "Gridsearch data reaches into the test season!"

    study_s1 = gs.run_stage1_study(variant_name, df, y, feat_cols, SEED, N_TRIALS_STAGE1, CUTOFF_Q, verbose=True,
                                    num_threads=LGB_NUM_THREADS, n_jobs=N_JOBS_TRIALS)
    stage1_best = {**study_s1.best_params, "rounds": study_s1.best_trial.user_attrs["best_round"]}
    study_s2 = gs.run_stage2_study(variant_name, df, y, feat_cols, stage1_best, SEED, N_TRIALS_STAGE2, CUTOFF_Q, verbose=True,
                                    num_threads=LGB_NUM_THREADS, n_jobs=N_JOBS_TRIALS)
    tuned_params = gs.pack(study_s1, study_s2)

    params_path = ROOT_DIR / f"best_params_season2025_26_holdout_{TARGET.lower()}_{variant_name}.json"
    with open(params_path, "w") as f:
        json.dump({variant_name: tuned_params}, f, indent=2)
    print(f"[{variant_name}] saved tuned params to {params_path}")

    return {
        "df": df, "y": y, "feat_cols": feat_cols,
        "study_s1": study_s1, "study_s2": study_s2,
        "stage1_best": stage1_best, "tuned_params": tuned_params,
    }


In [ ]:
gs3_results = {}
for variant_name, gt_file in GT_VARIANTS_3WAY.items():
    gs3_results[variant_name] = run_gridsearch_only(variant_name, gt_file)


### Summary table — best stage-1 delta-log L2 / stage-2 WIS per variant

In [ ]:
summary_table = pd.DataFrame({
    name: {
        "stage1_l2": r["study_s1"].best_value,
        "stage2_wis": r["study_s2"].best_value,
        "n_features": len(r["feat_cols"]),
        "n_rows": r["df"].shape[0],
    }
    for name, r in gs3_results.items()
}).T

if "no_gt" in summary_table.index:
    summary_table["stage1_l2_pct_vs_no_gt"] = 100 * (
        summary_table["stage1_l2"] - summary_table.loc["no_gt", "stage1_l2"]
    ) / summary_table.loc["no_gt", "stage1_l2"]
    summary_table["stage2_wis_pct_vs_no_gt"] = 100 * (
        summary_table["stage2_wis"] - summary_table.loc["no_gt", "stage2_wis"]
    ) / summary_table.loc["no_gt", "stage2_wis"]

summary_table


### Stage 1 — training vs validation learning curve, all 3 variants

Same technique as `Gridsearch_LightGBM.ipynb`'s stage-1 learning curve: trains once
per variant with its own tuned params out to a generous max round count (no early
stopping), tracking cases-space RMSE on both splits every round via a custom `feval`.
This is a diagnostic view in cases-units for interpretability — it's not the metric
`run_gridsearch_only` actually minimized (that's delta-log L2; see the summary table
above), so the two can disagree on exactly where "best" is. Solid = train, dashed =
valid, colored by variant.

In [ ]:
plt.figure(figsize=(10, 6))

for variant_name, r in gs3_results.items():
    df, y, feat_cols, stage1_best = r["df"], r["y"], r["feat_cols"], r["stage1_best"]

    cut = df["date"].quantile(CUTOFF_Q)
    tr_mask = (df["date"] <= cut).to_numpy()
    X_tr, y_tr = df.loc[tr_mask, feat_cols].astype(float), y[tr_mask]
    X_va, y_va = df.loc[~tr_mask, feat_cols].astype(float), y[~tr_mask]

    base_tr = np.log1p(np.clip(df.loc[tr_mask, "y_base"].to_numpy(float), 0, None))
    base_va = np.log1p(np.clip(df.loc[~tr_mask, "y_base"].to_numpy(float), 0, None))
    actual_tr = df.loc[tr_mask, "target"].to_numpy(float)
    actual_va = df.loc[~tr_mask, "target"].to_numpy(float)

    p1 = {
        "objective": "regression", "metric": "l2",
        "learning_rate": stage1_best["learning_rate"],
        "num_leaves": stage1_best["num_leaves"],
        "min_child_samples": stage1_best["min_child_samples"],
        "feature_fraction": stage1_best["feature_fraction"],
        "bagging_fraction": 0.9, "bagging_freq": 1,
        "seed": SEED, "deterministic": True, "force_row_wise": True,
        "verbosity": -1,
    }
    MAX_ROUNDS_S1 = max(1500, stage1_best["rounds"] * 2)

    d_tr = lgb.Dataset(X_tr, label=y_tr)
    d_va = lgb.Dataset(X_va, label=y_va, reference=d_tr)

    def _cases_rmse_feval(preds, dataset, _bt=base_tr, _bv=base_va, _at=actual_tr, _av=actual_va, _dtr=d_tr):
        base, actual = (_bt, _at) if dataset is _dtr else (_bv, _av)
        pred_cases = np.maximum(np.expm1(preds + base), 0.0)
        return "cases_rmse", float(np.sqrt(np.mean((pred_cases - actual) ** 2))), False

    evals_result = {}
    lgb.train(
        p1, d_tr,
        num_boost_round=MAX_ROUNDS_S1,
        valid_sets=[d_tr, d_va],
        valid_names=["train", "valid"],
        feval=_cases_rmse_feval,
        callbacks=[lgb.record_evaluation(evals_result)],
    )

    rounds = np.arange(1, MAX_ROUNDS_S1 + 1)
    color = COLORS_3WAY.get(variant_name, "#333333")
    plt.plot(rounds, evals_result["train"]["cases_rmse"], color=color, linestyle="-", alpha=0.85, label=f"{variant_name} (train)")
    plt.plot(rounds, evals_result["valid"]["cases_rmse"], color=color, linestyle="--", alpha=0.85, label=f"{variant_name} (valid)")
    print(f"[{variant_name}] stage-1 curve done ({MAX_ROUNDS_S1} rounds)")

plt.xlabel("boosting round")
plt.ylabel("RMSE (cases)")
plt.title(f"Stage 1 learning curve — {TARGET} — no_gt vs gt_proc vs gt_raw")
plt.legend()
plt.tight_layout()
plt.show()


### Stage 1 — same learning curve, in delta-log L2 (the metric actually optimized)

Same trained models as the cases-RMSE plot above, just reading LightGBM's native
`l2` metric directly (no custom `feval` needed here — `record_evaluation` already
tracks it). This is the metric `run_gridsearch_only`'s early stopping and Optuna
objective both actually use, so "best" on this plot lines up with the tuned `rounds`
in a way the cases-RMSE plot isn't guaranteed to.

In [ ]:
plt.figure(figsize=(10, 6))

for variant_name, r in gs3_results.items():
    df, y, feat_cols, stage1_best = r["df"], r["y"], r["feat_cols"], r["stage1_best"]

    cut = df["date"].quantile(CUTOFF_Q)
    tr_mask = (df["date"] <= cut).to_numpy()
    X_tr, y_tr = df.loc[tr_mask, feat_cols].astype(float), y[tr_mask]
    X_va, y_va = df.loc[~tr_mask, feat_cols].astype(float), y[~tr_mask]

    p1 = {
        "objective": "regression", "metric": "l2",
        "learning_rate": stage1_best["learning_rate"],
        "num_leaves": stage1_best["num_leaves"],
        "min_child_samples": stage1_best["min_child_samples"],
        "feature_fraction": stage1_best["feature_fraction"],
        "bagging_fraction": 0.9, "bagging_freq": 1,
        "seed": SEED, "deterministic": True, "force_row_wise": True,
        "verbosity": -1,
    }
    MAX_ROUNDS_S1 = max(1500, stage1_best["rounds"] * 2)

    d_tr = lgb.Dataset(X_tr, label=y_tr)
    d_va = lgb.Dataset(X_va, label=y_va, reference=d_tr)

    evals_result = {}
    lgb.train(
        p1, d_tr,
        num_boost_round=MAX_ROUNDS_S1,
        valid_sets=[d_tr, d_va],
        valid_names=["train", "valid"],
        callbacks=[lgb.record_evaluation(evals_result)],
    )

    rounds = np.arange(1, MAX_ROUNDS_S1 + 1)
    color = COLORS_3WAY.get(variant_name, "#333333")
    plt.plot(rounds, evals_result["train"]["l2"], color=color, linestyle="-", alpha=0.85, label=f"{variant_name} (train)")
    plt.plot(rounds, evals_result["valid"]["l2"], color=color, linestyle="--", alpha=0.85, label=f"{variant_name} (valid)")
    plt.axvline(stage1_best["rounds"], color=color, linestyle=":", alpha=0.5)
    print(f"[{variant_name}] stage-1 L2 curve done ({MAX_ROUNDS_S1} rounds, tuned rounds={stage1_best['rounds']})")

plt.xlabel("boosting round")
plt.ylabel("delta-log L2 (MSE)")
plt.title(f"Stage 1 learning curve (delta-log L2) — {TARGET} — no_gt vs gt_proc vs gt_raw")
plt.legend()
plt.tight_layout()
plt.show()


### Stage 2 — training vs validation learning curve, all 3 variants

Same round-grid-refit technique as `Gridsearch_LightGBM.ipynb` (LightGBMLSS doesn't
expose a per-round WIS during training) — refits stage1+stage2 at each `stage2_rounds`
grid point per variant, scores WIS on train/valid. Slower than stage 1 (retrains a
model per grid point per variant, 3 × 13 ≈ 39 fits) — narrow `round_grid` below if
it's too slow.

In [ ]:
plt.figure(figsize=(10, 6))

for variant_name, r in gs3_results.items():
    df, y, feat_cols = r["df"], r["y"], r["feat_cols"]
    stage1_best, s2_params = r["stage1_best"], r["study_s2"].best_params

    cut = df["date"].quantile(CUTOFF_Q)
    tr_mask = (df["date"] <= cut).to_numpy()
    X_tr, y_tr = df.loc[tr_mask, feat_cols].astype(float), y[tr_mask]
    X_va = df.loc[~tr_mask, feat_cols].astype(float)
    base_tr = df.loc[tr_mask, "y_base"].to_numpy(float)
    base_va = df.loc[~tr_mask, "y_base"].to_numpy(float)
    actual_tr = df.loc[tr_mask, "target"].to_numpy(float)
    actual_va = df.loc[~tr_mask, "target"].to_numpy(float)
    ok_tr = np.isfinite(actual_tr)
    ok_va = np.isfinite(actual_va)

    tuned_rounds = s2_params["stage2_rounds"]
    round_grid = sorted(set(list(range(10, 261, 20)) + [tuned_rounds]))

    train_wis, valid_wis = [], []
    for rr in round_grid:
        stage1_r, stage2_r = gs.fit_two_stage_one_bag(
            X_train=X_tr, y_train=y_tr,
            stage1_rounds=stage1_best["rounds"], seed=SEED,
            num_leaves=stage1_best["num_leaves"], learning_rate=stage1_best["learning_rate"],
            min_child_samples=stage1_best["min_child_samples"], feature_fraction=stage1_best["feature_fraction"],
            stage2_rounds=rr, sigma_mode="bounded",
            lambda_l2=s2_params["lambda_l2"], s2_min_child_samples=s2_params["s2_min_child_samples"],
            s2_num_leaves=None, s2_learning_rate=None, s2_feature_fraction=None, s2_max_depth=6,
        )

        pq_tr = gs.predict_quantiles(stage1_r, stage2_r, X_tr, gs.QUANTILES, target_mode="delta_log", current_obs=base_tr)
        pq_tr = np.clip(np.nan_to_num(pq_tr, nan=0.0, posinf=1e7, neginf=0.0), 0.0, 1e7)
        train_wis.append(float(np.mean(gs.wis_vectorized(pq_tr[ok_tr], actual_tr[ok_tr], gs.QUANTILES))))

        pq_va = gs.predict_quantiles(stage1_r, stage2_r, X_va, gs.QUANTILES, target_mode="delta_log", current_obs=base_va)
        pq_va = np.clip(np.nan_to_num(pq_va, nan=0.0, posinf=1e7, neginf=0.0), 0.0, 1e7)
        valid_wis.append(float(np.mean(gs.wis_vectorized(pq_va[ok_va], actual_va[ok_va], gs.QUANTILES))))

    color = COLORS_3WAY.get(variant_name, "#333333")
    plt.plot(round_grid, train_wis, marker="o", color=color, linestyle="-", alpha=0.85, label=f"{variant_name} (train)")
    plt.plot(round_grid, valid_wis, marker="o", color=color, linestyle="--", alpha=0.85, label=f"{variant_name} (valid)")
    print(f"[{variant_name}] stage-2 curve done ({len(round_grid)} grid points)")

plt.xlabel("stage-2 boosting rounds")
plt.ylabel("WIS")
plt.title(f"Stage 2 learning curve — {TARGET} — no_gt vs gt_proc vs gt_raw")
plt.legend()
plt.tight_layout()
plt.show()
